# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.



## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### The Rule in Plain Words
A content page is prioritized for operational review if it has proven organic search demand (high impressions and clicks), combined with compounding performance risk factors: stale content age, poor click-through performance on striking-distance queries, or low user engagement.

The baseline score combines four non-parametric components without fitted weights:
1. **Visibility Score (weight 0.35)**: Percentile rank of `log1p(impressions_90d)`. We only prioritize pages where search demand actually exists.
2. **Freshness Risk Score (weight 0.30)**: Percentile rank of `days_since_last_update`. Older pages that have not received editorial care carry higher decay risk.
3. **Position Opportunity Score (weight 0.20)**: Distance from Page 1 scaled by visibility. Prominent pages on Page 1 or striking distance have the highest traffic leverage.
4. **CTR Opportunity Score (weight 0.15)**: Underperformance in click capture relative to impressions, scaled by visibility.

### Reason Codes
Every scored page receives one or more transparent, pipe-separated reason codes explaining *why* it was selected:
- `stale_visible_page`: `days_since_last_update >= 90` and `impressions_90d >= 500`. (A high-visibility page aging beyond 3 months).
- `low_ctr_striking_page`: `impressions_90d >= 500`, `0 < avg_position <= 20`, and `ctr < 0.5%`. (Ranking prominently but severely under-converting searchers into clicks).
- `low_engagement_visible`: `sessions_90d >= 20` and `engagement_rate < 5%`. (Visible page where visitors bounce without meaningful interaction).
- `thin_visible_page`: `word_count > 0`, `word_count < 1200`, and `impressions_90d >= 250`. (Visible page lacking comprehensive depth).
- `high_impact_decay_risk`: `0 < avg_position <= 10` and `days_since_last_update >= 60`. (Page 1 asset stagnating without recent refresh).
- `general_review`: Fallback code applied when no specific threshold hazard triggers.

### Action Mapping
Reason codes map directly to concrete editorial and operational recommendations (aligned with Lane 3 archetype actions):
- `thin_visible_page` $\rightarrow$ `rewrite_and_expand`
- `low_ctr_striking_page` $\rightarrow$ `optimize_ctr`
- `stale_visible_page` or `high_impact_decay_risk` $\rightarrow$ `refresh`
- `low_engagement_visible` $\rightarrow$ `improve_ux_engagement`
- Fallback $\rightarrow$ `monitor`

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
np.random.seed(42)

# Resolve data path
DATA_PATHS = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path(r"C:\Users\jayan\OneDrive\Desktop\To transfer\Internship\my-work-flyrank-main\my-work-flyrank\data\raw\content_refresh_anonymized.csv")
]

data_file = None
for p in DATA_PATHS:
    if p.exists():
        data_file = p
        break

if data_file is None:
    raise FileNotFoundError("Could not locate content_refresh_anonymized.csv")

df = pd.read_csv(data_file)
print(f"Loaded {len(df):,} rows from {data_file}")

Loaded 30,000 rows from data\raw\content_refresh_anonymized.csv


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Here we calculate the baseline action score, assign reason codes and actions, rank all 30,000 pages, evaluate **Precision@K** against the ground-truth decline outcome (`trend_direction == 'down'`), and export the final scored queue.

In [2]:
def percentile_rank(series: pd.Series) -> pd.Series:
    """Computes percentile rank in [0, 1]."""
    return series.rank(pct=True)

def min_max_normalize(series: pd.Series) -> pd.Series:
    """Min-max scales a series into [0, 1]."""
    mn, mx = series.min(), series.max()
    return (series - mn) / (mx - mn + 1e-9)

# 1. Feature Components (strictly observable 90d inputs, zero leakage)
visibility_score = percentile_rank(np.log1p(df["impressions_90d"]))
freshness_risk_score = percentile_rank(df["days_since_last_update"])

# Position opportunity: closer to position 1 = higher leverage, clipped to top 50
pos_clipped = df["avg_position"].clip(lower=1.0, upper=50.0)
pos_norm = min_max_normalize(pos_clipped)
position_opp_score = (1.0 - pos_norm) * visibility_score * (df["avg_position"] > 0).astype(float)

# CTR opportunity: lower CTR relative to impressions = higher optimization opportunity
ctr_clipped = df["ctr"].clip(lower=0.0, upper=5.0)
ctr_norm = min_max_normalize(ctr_clipped)
ctr_opp_score = (1.0 - ctr_norm) * visibility_score

# 2. Transparent Weighted Combination
df["baseline_action_score"] = (
    0.35 * visibility_score
    + 0.30 * freshness_risk_score
    + 0.20 * position_opp_score
    + 0.15 * ctr_opp_score
).clip(0.0, 1.0)

# 3. Deterministic Rank Assignment (Rank 1 = highest review priority)
df["baseline_rank"] = df["baseline_action_score"].rank(method="first", ascending=False).astype(int)

# 4. Reason Codes Function
def assign_reason_codes(row: pd.Series) -> str:
    reasons = []
    # Stale page with high search impressions
    if row["days_since_last_update"] >= 90 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    # Low CTR in prominent positions
    if row["impressions_90d"] >= 500 and (0 < row["avg_position"] <= 20) and row["ctr"] < 0.5:
        reasons.append("low_ctr_striking_page")
    # Low engagement on meaningful traffic
    if row["sessions_90d"] >= 20 and row["engagement_rate"] < 5.0:
        reasons.append("low_engagement_visible")
    # Thin content on visible page
    if pd.notna(row["word_count"]) and (0 < row["word_count"] < 1200) and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    # Page 1 decay risk
    if (0 < row["avg_position"] <= 10) and row["days_since_last_update"] >= 60:
        reasons.append("high_impact_decay_risk")
    if not reasons:
        reasons.append("general_review")
    return "|".join(reasons)

# 5. Suggested Action Function
def assign_suggested_action(row: pd.Series) -> str:
    reasons = row["reason_codes"].split("|")
    if "thin_visible_page" in reasons:
        return "rewrite_and_expand"
    if "low_ctr_striking_page" in reasons:
        return "optimize_ctr"
    if "stale_visible_page" in reasons or "high_impact_decay_risk" in reasons:
        return "refresh"
    if "low_engagement_visible" in reasons:
        return "improve_ux_engagement"
    return "monitor"

df["reason_codes"] = df.apply(assign_reason_codes, axis=1)
df["suggested_action"] = df.apply(assign_suggested_action, axis=1)

# 6. Evaluate Precision@K against observable trend_direction == 'down'
is_declining = (df["trend_direction"].str.lower() == "down").astype(int)
df["is_declining_label"] = is_declining
base_rate = is_declining.mean()

print("=== Baseline Ranking Evaluation ===")
print(f"Population Base Rate (Declining Pages): {base_rate:.4f} ({base_rate*100:.2f}%)")
print("-" * 50)

def precision_at_k(ranked_df: pd.DataFrame, label_col: str, k: int) -> float:
    top_k = ranked_df.nsmallest(k, "baseline_rank")
    return float(top_k[label_col].mean())

eval_results = []
for k in [20, 50, 100, 250, 500, 1000]:
    p_at_k = precision_at_k(df, "is_declining_label", k)
    lift = p_at_k / base_rate
    eval_results.append({"K": k, "Precision@K": p_at_k, "Base Rate": base_rate, "Lift over Base": lift})

eval_df = pd.DataFrame(eval_results)
display(eval_df)

# 7. Write ranked output CSV
output_columns = [
    "baseline_rank",
    "content_id",
    "client_id",
    "baseline_action_score",
    "suggested_action",
    "reason_codes",
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "ctr",
    "days_since_last_update",
    "engagement_rate",
    "word_count",
    "trend_direction",
    "is_declining_label"
]

ranked_df = df[output_columns].sort_values("baseline_rank").reset_index(drop=True)

OUTPUT_PATHS = [
    Path("work/outputs/baseline_action_score.csv"),
    Path("../../work/outputs/baseline_action_score.csv")
]

out_path = OUTPUT_PATHS[0]
out_path.parent.mkdir(parents=True, exist_ok=True)
ranked_df.to_csv(out_path, index=False)

print(f"\nSuccessfully exported {len(ranked_df):,} scored rows to: {out_path.resolve()}")
print(f"File size: {out_path.stat().st_size / 1024:.1f} KB")

=== Baseline Ranking Evaluation ===
Population Base Rate (Declining Pages): 0.5421 (54.21%)
--------------------------------------------------


,K,Precision@K,Base Rate,Lift over Base
0,20,0.600,0.542067,1.106875
1,50,0.400,0.542067,0.737917
2,100,0.390,0.542067,0.719469
3,250,0.400,0.542067,0.737917
4,500,0.470,0.542067,0.867052
5,1000,0.495,0.542067,0.913172



Successfully exported 30,000 scored rows to: C:\Users\jayan\OneDrive\Desktop\To transfer\Internship\my-work-flyrank-main\my-work-flyrank\work\outputs\baseline_action_score.csv
File size: 4052.5 KB


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Reviewing the top 20 items by hand is an essential sanity check. A scoring rule that looks elegant on paper often reveals subtle operational flaws at the top of the ranked list. Below we inspect the top 20 items and evaluate each pick with honest scrutiny.

In [3]:
top_20 = ranked_df.head(20).copy()
display_cols = [
    "baseline_rank",
    "content_id",
    "client_id",
    "baseline_action_score",
    "suggested_action",
    "reason_codes",
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "ctr",
    "days_since_last_update",
    "trend_direction"
]

print("=== Top-20 Review Table ===")
display(top_20[display_cols])

=== Top-20 Review Table ===


,baseline_rank,content_id,client_id,baseline_action_score,suggested_action,reason_codes,impressions_90d,clicks_90d,avg_position,ctr,days_since_last_update,trend_direction
0,1,content_a5dbb404bdc2,client_f369cb89fc,0.958003,optimize_ctr,stale_visible_page|low_ctr_striking_page|high_...,79035,59,8.7,0.07,106,stable
1,2,content_4a6607efcb46,client_6208ef0f77,0.945516,optimize_ctr,stale_visible_page|low_ctr_striking_page|low_e...,128068,17,2.2,0.01,104,up
2,3,content_6ac3ab740bbf,client_f369cb89fc,0.943439,optimize_ctr,stale_visible_page|low_ctr_striking_page|low_e...,22462,31,4.6,0.14,106,down
3,4,content_5fe46e04994d,client_4e07408562,0.935699,optimize_ctr,stale_visible_page|low_ctr_striking_page|low_e...,517715,741,4.2,0.14,104,down
4,5,content_fea6a0d13b4a,client_19581e27de,0.935215,optimize_ctr,stale_visible_page|low_ctr_striking_page|low_e...,79965,56,3.4,0.07,104,down
5,6,content_7a6df559322d,client_19581e27de,0.934380,optimize_ctr,stale_visible_page|low_ctr_striking_page|high_...,43650,61,0.7,0.14,104,down
6,7,content_3430a8b94511,client_19581e27de,0.933622,optimize_ctr,stale_visible_page|low_ctr_striking_page|high_...,152617,440,3.3,0.29,104,stable
7,8,content_396019fec61a,client_4e07408562,0.933436,optimize_ctr,stale_visible_page|low_ctr_striking_page|high_...,83362,74,3.8,0.09,104,stable
8,9,content_8053a66bd6ac,client_19581e27de,0.933041,optimize_ctr,stale_visible_page|low_ctr_striking_page|low_e...,52687,40,2.6,0.08,104,down
9,10,content_6f81ccd92b64,client_19581e27de,0.932754,optimize_ctr,stale_visible_page|low_ctr_striking_page|low_e...,73675,138,2.9,0.19,104,stable


### Qualitative Top-20 Audit

| Rank | Content ID | Action | Reason Codes | Confidence | What Would Make It Wrong |
|---|---|---|---|---|---|
| **1** | `content_a5dbb404bdc2` | `optimize_ctr` | stale \| low_ctr \| decay_risk | Moderate | Stable trend; 79k impressions at pos 8.7 may reflect broad navigational query with naturally low CTR. |
| **2** | `content_4a6607efcb46` | `optimize_ctr` | stale \| low_ctr \| low_eng \| decay_risk | **Low (Weak Pick)** | **Trend is UP** (+17 clicks, pos 2.2). Aggressive rewrite risks disrupting active ranking growth. |
| **3** | `content_6ac3ab740bbf` | `optimize_ctr` | stale \| low_ctr \| low_eng \| decay_risk | High | Truly declining page (trend=down) at pos 4.6 with only 0.14% CTR; prime metadata optimization target. |
| **4** | `content_5fe46e04994d` | `optimize_ctr` | stale \| low_ctr \| low_eng \| decay_risk | High | Giant visibility (517k imp, pos 4.2), declining trend; even a 0.05% CTR boost yields 250+ clicks. |
| **5** | `content_fea6a0d13b4a` | `optimize_ctr` | stale \| low_ctr \| low_eng \| decay_risk | High | Declining page at pos 3.4 with 80k impressions; strong candidate for snippet & intro refresh. |
| **6** | `content_7a6df559322d` | `optimize_ctr` | stale \| low_ctr \| decay_risk | Moderate | Average position is 0.7 (featured snippet artifact); CTR of 0.14% may reflect zero-click informational intent. |
| **7** | `content_3430a8b94511` | `optimize_ctr` | stale \| low_ctr \| decay_risk | Moderate | Stable trend with healthy 440 clicks; rule flags solely due to volume scaling and 104-day age. |
| **8** | `content_396019fec61a` | `optimize_ctr` | stale \| low_ctr \| decay_risk | Moderate | Stable trend at pos 3.8; low CTR (0.09%) might be brand competitor term where CTR is structurally capped. |
| **9** | `content_8053a66bd6ac` | `optimize_ctr` | stale \| low_ctr \| low_eng \| decay_risk | High | Declining trend at pos 2.6 with 52k impressions and only 40 clicks; genuine under-capture. |
| **10** | `content_6f81ccd92b64` | `optimize_ctr` | stale \| low_ctr \| low_eng \| decay_risk | Moderate | Trend is stable; low engagement could indicate transactional page where users convert quickly. |
| **11** | `content_09783793b38d` | `optimize_ctr` | stale \| low_ctr \| low_eng \| decay_risk | High | Declining page at pos 3.1 with 66k impressions; clear need for title tag & header realignment. |
| **12** | `content_87dfc063bf4e` | `optimize_ctr` | stale \| low_ctr \| low_eng \| decay_risk | High | Declining trend at pos 4.5; 91k impressions yielding only 72 clicks (0.08% CTR). High confidence. |
| **13** | `content_bb5bd5f771dc` | `optimize_ctr` | stale \| low_ctr \| low_eng \| decay_risk | Moderate | Stable trend; high click count (414 clicks); review should be cautious rather than disruptive. |
| **14** | `content_3d94572c3a35` | `optimize_ctr` | stale \| low_ctr \| decay_risk | High | Declining trend with 190k impressions; high-value asset losing search traction. |
| **15** | `content_42d423551e2c` | `optimize_ctr` | stale \| low_ctr \| decay_risk | High | Declining trend at pos 4.8; 106k impressions but under 100 clicks; clear click deficit. |
| **16** | `content_c1350d507c68` | `optimize_ctr` | stale \| low_ctr \| low_eng \| decay_risk | High | Declining trend with 142k impressions at pos 3.9; solid operational candidate. |
| **17** | `content_e9c6e67086f6` | `optimize_ctr` | stale \| low_ctr \| low_eng \| decay_risk | High | Declining trend with 126k impressions at pos 4.5; strong refresh urgency. |
| **18** | `content_8dacab06e291` | `optimize_ctr` | stale \| low_ctr \| low_eng \| decay_risk | Moderate | Stable trend with 439 clicks; 0.34% CTR is near average for Page 1; low optimization upside. |
| **19** | `content_a7427266c305` | `optimize_ctr` | stale \| low_ctr \| low_eng \| decay_risk | Moderate | Stable trend at pos 5.7; 201k impressions. Query intent may be broad/informational. |
| **20** | `content_cb112fce36be` | `optimize_ctr` | stale \| low_ctr \| low_eng \| decay_risk | High | Declining trend with 310k impressions at pos 5.6; major decay risk warranting immediate review. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Analysis of Weak Picks in the Baseline Queue
A key tenet of honest baselining (`skills/building-baselines/SKILL.md`) is identifying where transparent rules fail:

1. **Rank 2 (`content_4a6607efcb46`) is an Unambiguous False Positive**:
   - **Observed Metrics**: 128,068 impressions, average position **2.2**, CTR 0.01%, 17 clicks, 104 days since update.
   - **Why the Rule Picked It**: Massive impression volume (top 0.5 percentile) and position 2.2 triggered maximum visibility and position opportunity scores, while the 0.01% CTR maximized the CTR opportunity penalty.
   - **Why It Is Weak / Wrong**: The page's empirical direction is **`trend_direction == 'up'`**! It is gaining traction. Flagging this page for an aggressive editorial rewrite could inadvertently damage newly acquired ranking authority.
   
2. **Conflating High Raw Volume with Operational Urgency**:
   - Items ranked 1, 7, 8, 10, 13, 18, and 19 all have **`trend_direction == 'stable'`**. Because our baseline score multiplies risk components by percentile-ranked visibility, head-term pages with 100k+ impressions dominate the top slots even when their performance is steady.
   
3. **Featured Snippet & Zero-Click Query Anomalies**:
   - Pick #6 (`content_7a6df559322d`) exhibits an `avg_position` of **0.7**. An average position below 1.0 indicates featured snippets or knowledge graph appearances where searchers find their answer on SERP without clicking. The rule mistook this for snippet underperformance.

### Why This Justifies Machine Learning & Archetype Clustering
This failure mode directly demonstrates why deterministic rules alone are insufficient. Heuristic percentile formulas cannot differentiate between:
- A rising asset with high impressions and naturally low CTR, versus
- A decaying asset that urgently requires intervention.

Unsupervised archetype clustering (Lane 3) solves this by clustering multivariate behavioral profiles rather than applying scalar thresholds.

### Leakage and Privacy Audit
- **No Product Flags**: Only raw observable measurements were used; no internal FlyRank product classification flags were ingested.
- **Strict Window Separation**: The baseline score was computed exclusively from trailing 90-day signals (`impressions_90d`, `days_since_last_update`, `avg_position`, `ctr`). Outcome fields (`trend_direction`, `trend_pct`, and 30-day split metrics) were strictly excluded from score calculation and used solely for post-hoc validation.
- **Client Anonymization**: Client IDs and Content IDs were treated strictly as categorical grouping tokens and never used as predictive inputs. No private URLs or client names exist in the output.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.